# C9-dimensionality-reduction — Practice p17 — Solution

In [1]:
import numpy as np

SEED = 20260804
rng = np.random.default_rng(SEED)
n_per = 200
a1 = rng.uniform(0.0, np.pi, n_per)
a2 = rng.uniform(0.0, np.pi, n_per)
moon0 = np.column_stack([np.cos(a1), np.sin(a1)])
moon1 = np.column_stack([1.0 - np.cos(a2), 0.5 - np.sin(a2)])
X = np.vstack([moon0, moon1]) + rng.normal(0.0, 0.04, (2 * n_per, 2))
labels = np.repeat([0, 1], n_per)

delta = X[:, None, :] - X[None, :, :]
D2 = (delta * delta).sum(axis=2)
D = np.sqrt(D2)
D2_rank = D2.copy()
np.fill_diagonal(D2_rank, np.inf)

def graph_summary(k):
    neighbors = np.argsort(D2_rank, axis=1)[:, :k]
    directed = np.zeros((400, 400), dtype=bool)
    directed[np.arange(400)[:, None], neighbors] = True
    adjacency = directed | directed.T
    reachability = adjacency | np.eye(400, dtype=bool)
    for _ in range(9):
        reachability = reachability @ reachability
    edges = int(np.triu(adjacency, 1).sum())
    cross = int(np.triu(adjacency & (labels[:, None] != labels[None, :]), 1).sum())
    components = int(np.unique(reachability, axis=0).shape[0])
    return adjacency, edges, cross, components

Adj, n_edges, n_cross, n_components = graph_summary(6)
_, n_edges_k4, n_cross_k4, n_components_k4 = graph_summary(4)
D_no_self = D.copy()
np.fill_diagonal(D_no_self, np.inf)
median_nn = float(np.median(np.min(D_no_self, axis=1)))
min_cross = float(np.min(D[labels[:, None] != labels[None, :]]))

print("k=6 edges / cross / components:", n_edges, n_cross, n_components)
print("k=4 edges / cross / components:", n_edges_k4, n_cross_k4, n_components_k4)
print("median nearest / minimum cross distance:", median_nn, min_cross)

k=6 edges / cross / components: 1468 0 2
k=4 edges / cross / components: 1020 0 6
median nearest / minimum cross distance: 0.023096476293121296 0.3365588679084602


The median nearest-neighbor distance is $0.0230965$, far below the minimum cross-moon distance $0.3365589$, so six nearby same-moon choices connect each crescent while making no cross edges; the graph therefore has two components, and a layout would show those local components but would not make their global separation quantitative.  At $k=4$, the sparser graph fragments into six components because some within-moon gaps are not bridged.  As $k$ keeps growing, cross edges must eventually appear and the component count must ultimately fall to one.

### Answer check

In [2]:
assert Adj.shape == (400, 400) and Adj.dtype == bool
assert np.array_equal(Adj, Adj.T)
assert n_edges == 1468 and n_cross == 0 and n_components == 2
assert n_edges_k4 == 1020 and n_cross_k4 == 0 and n_components_k4 == 6
assert np.isclose(median_nn, 0.023096476293121296, atol=1e-12, rtol=0)
assert np.isclose(min_cross, 0.3365588679084602, atol=1e-12, rtol=0)